### Construct a scRNA-seq reference

Data from fourteen patients pathologically diagnosed with ovarian cancer. Fresh samples including primary ovarian tumour, omentum metastatic tumour, pelvic lymph node, malignant ascites and peripheral blood were obtained from these patients during surgery. (https://doi.org/10.1038/s43018-023-00599-8). Single-cell gene expression and immune repertoire measurements were conducted using the Chromium Single Cell V(D)J Reagent Kit. Completed libraries were sequenced on an Illumina NovaSeq6000 system.

In [ ]:
import os
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
WORKING_DIR = "."
DATA_DIR = "data"
ADATA = f"{DATA_DIR}/OvC_adata_zheng.h5ad"
OUT_DIR = os.path.join(WORKING_DIR, "outs")
if not os.path.exists(OUT_DIR):
    os.makedirs(OUT_DIR)

sc.settings.figdir = OUT_DIR

In [ ]:
adata_sc = sc.read_h5ad(ADATA)
adata_sc

In [ ]:
adata_sc.obs['Patients'].value_counts()

In [ ]:
adata_sc = adata_sc[adata_sc.obs["Patients"].str.startswith('HGSOC')].copy()
adata_sc = adata_sc[adata_sc.obs['Groups'].isin(['Primary Tumor', 'Metastatic Tumor'])].copy()

In [ ]:
adata_sc.obs['Patients'].value_counts()

In [ ]:
adata_sc.obs['maintypes_2'].value_counts()

In [ ]:
adata_sc.obs['cell_type'] = adata_sc.obs['maintypes_2'].replace({
    'Epithelial cells': 'Cancer epithelial cells',
    'NK': 'Natural killer cells',
    'B': 'B cells',
    'Macrophage': 'Macrophages',
    'CD3+ T': 'CD3+ T cells',
    'CD8+ T': 'CD8+ T cells'
})
adata_sc = adata_sc[~adata_sc.obs['cell_type'].isin(['DC', 'HSC', 'Monocyte', 'Mesothelial cells'])].copy()
adata_sc.obs['cell_type'].value_counts()

In [ ]:
adata_sc.obs['total_counts'] = adata_sc.X.sum(axis=1)
plt.figure(figsize=(8, 5))
sns.histplot(adata_sc.obs['total_counts'], kde=True, bins=50)
plt.xlabel("Total Transcript Counts per Cell")
plt.ylabel("Number of Cell")
plt.title("Distribution of Total Transcript Counts per Cell")
plt.show()

In [ ]:
adata_sc.obs['n_genes'] = (adata_sc.X > 0).sum(axis=1)
plt.figure(figsize=(8, 5))
sns.histplot(adata_sc.obs['n_genes'], kde=True, bins=50)
plt.xlabel("Number of Genes Detected per Cell")
plt.ylabel("Number of Cells")
plt.title("Distribution of Genes Detected per Cell")
plt.show()

In [ ]:
adata_sc.obs.Groups.value_counts()

In [ ]:
adata_sc.n_obs

In [ ]:
adata_sc.var

In [ ]:
adata_sc.write_h5ad(f"{DATA_DIR}/OvC_adata_zheng_preprocessed.h5ad")